# 03 · Check every drawing against the code rules: the compliance judge (`ai_query`)

Third and last step of the document-intelligence chain (parse, extract and classify, then check compliance), and the hero AI step of the demo. For each of the 1,200 drawings a Claude model (`databricks-claude-sonnet-4-5`, called through `ai_query`) reads the drawing's parsed text against five engineering rules and lists the violations it finds. The notebook writes **`mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings`**: one row per **finding**, so a drawing with three violations has three rows.

| column | type | what it holds |
|---|---|---|
| `drawing_id`, `finding_id` | STRING | primary key together; `finding_id` is `<drawing_id>-C01`, `-C02`, ... (or `-PASS`, `-REVIEW`) |
| `volume_path` | STRING | where the PDF is, from `drawings_parsed` |
| `page_num` | INT | 0-based page of the violation; NULL on PASS rows |
| `bbox` | `ARRAY<DOUBLE>` | `[x, y, w, h]` as fractions (0-1) of the page, origin top-left; NULL on PASS rows |
| `bbox_source` | STRING | where the box came from: `ground_truth`, `parse_element` or `page` (see below); NULL on PASS rows |
| `discipline`, `revision`, `drawing_type` | STRING | from `drawings_extracted` |
| `status` | STRING | `PASS`, `FAIL` or `REVIEW` |
| `severity` | STRING | `critical`, `high`, `medium` or `low`; anything else the model says becomes `medium`; NULL on PASS rows |
| `reason` | STRING | one or two sentences a project manager can read |
| `citation` | STRING | one of the five rules' citations on FAIL rows; NULL otherwise |
| `evidence_text` | STRING | the model's short quote from the drawing text that shows the violation |
| `matched_ground_truth` | BOOLEAN | the drawing really has a planted flaw with this citation (left join to `ground_truth_findings`); NULL on PASS and REVIEW rows |
| `review_date` | DATE | simulated review date (see below) |
| `generated_at` | TIMESTAMP | when the judge ran |

**Row rules.** Every drawing has at least one row. A drawing with no violation gets exactly one `PASS` row (citation, page, box and severity NULL); a drawing with N violations gets N `FAIL` rows and no PASS row; a drawing whose judge answer is an error or cannot be parsed gets exactly one `REVIEW` row with a reason. "Reviewed / passed / failed" is therefore computable from this table alone.

**How it runs.**

- The judge sees only the drawing's `parsed_markdown`, its `drawing_type` and `discipline`, and one fixed rule set (printed below). It is never given the planted-flaw ground truth, the manifest's `has_flaw`, or any generator detail. The five rules are paraphrased engineering requirements with their clause numbers: ASME and IBC are copyrighted, so no code text is quoted.
- The AI step runs once, in the cell that writes `compliance_judge_raw`: the answer for every drawing, exactly as `ai_query` returned it. Everything after it (flattening, boxes, review dates, the findings table, the checks and the sample) reads that table, so no drawing is sent to the model twice and changing a rule of the clean-up costs nothing. The parsing and flattening logic is `compliance_lib.py`, next to this notebook, unit-tested locally (`03_doc_intelligence/tests/test_findings_grain.py`).
- Running the notebook again sends all 1,200 drawings to Sonnet again (about 1,200 calls, several million input tokens) and replaces both tables. The job that runs it must switch off serverless auto-retry (`disable_auto_optimization`, see `docs/BUILD.md`).
- A positive `max_rows` widget sends only that many drawings (half of them drawings the manifest lists as flawed, so a trial shows something) and writes `*_dryrun` tables instead, so a trial run cannot replace the real tables.
- The answers come from a model, so a re-run can differ a little from the run recorded here.

In [1]:
import contextlib
import io
import json
import os
import re
import sys
import time
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType

sys.path.insert(0, os.getcwd())  # a notebook's working directory is its own folder, where compliance_lib.py lives
import compliance_lib as lib

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
# 0 sends every drawing to the model. A positive number sends only that many (a fixed subset): a trial run that writes
# <table>_dryrun, so it can never replace the real tables.
dbutils.widgets.text("max_rows", "0", "Max drawings to send to the model (0 = all)")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
MAX_ROWS = int(dbutils.widgets.get("max_rows") or 0)
SUFFIX = "_dryrun" if MAX_ROWS else ""

SILVER = f"{catalog}.{prefix}_silver_engineering"
GOLD = f"{catalog}.{prefix}_gold_pm"
PARSED = f"{SILVER}.drawings_parsed"
EXTRACTED = f"{SILVER}.drawings_extracted"
GROUND_TRUTH_TABLE = f"{SILVER}.ground_truth_findings"
RAW = f"{GOLD}.compliance_judge_raw{SUFFIX}"
TARGET = f"{GOLD}.compliance_findings{SUFFIX}"

MODEL = "databricks-claude-sonnet-4-5"
PARTITIONS = 32  # tasks that make the AI calls side by side
SAMPLE_ID = "BRL-INS-PID-0052"  # the P&ID with three planted flaws that the parse notebook wrote out in full
MANIFEST = Path("../01_synthetic_data/outputs/drawing_manifest.json")  # same relative path in the repo and in a bundle deploy
GROUND_TRUTH = Path("../01_synthetic_data/outputs/ground_truth_findings.jsonl")
OUT = Path("outputs")  # next to this notebook


def sql_literal(text):
    # `text` as a SQL string literal (backslashes and single quotes escaped).
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


print(f"source: {PARSED} + {EXTRACTED}")
print(f"the model's answers: {RAW}")
print(f"target: {TARGET}")
print(f"drawings to send: {MAX_ROWS or 'all'}" + (" (trial run: the real tables are not touched)" if MAX_ROWS else ""))

source: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_parsed + mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted
the model's answers: mfg_mc_se_sa.bluerridge_gold_pm.compliance_judge_raw
target: mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings
drawings to send: all


## The rules and the prompt

**How the prompt got to this form (one revision, kept for the record).** The first full run used a plainer prompt and answer format. It found 97.8% of the 274 planted flaws but also reported hundreds of violations that the drawing text disproves (a LOTO tag or PSV that is in the text, sometimes beside a reason saying so): 578 of 1,200 drawings failed, 367 of 373 P&IDs among them, and only 29.8% of the distinct (drawing, citation) findings were planted flaws. The prompt was revised once, not tuned in a loop: it now tells the judge to search the whole text for the item that would make the drawing comply before reporting it missing, spells out how tags map to equipment (suffix letters, `O`/`0`, vessels are not powered equipment, a schematic needs no orifice), and the answer format gets two fields written *before* the verdict, `expected_item` and `found_in_text`, so the judge states what it looked for and whether it found it. `compliance_lib` drops an entry whose `found_in_text` is true. The revision was tried on a sample of 48 drawings first (recall 100%, precision 81%, against 100% and 33% for the first prompt on the same drawings).

Five rules, one per citation. Each is a paraphrased requirement that a reviewer could check from the drawing text alone, written the way the requirement reads on a real project. The prompt is the instructions, the five rules and the answer format (the same text for every drawing), followed by the three per-drawing inputs: the type, the discipline and the parsed text. Printed in full so the inputs can be audited.

In [2]:
PREFIX_TEXT = lib.build_prompt_prefix()
print(PREFIX_TEXT)
print("--- per drawing, appended after the text above:")
print("DRAWING TYPE: <drawing_type>\nDISCIPLINE: <discipline>\n\nDRAWING TEXT:\n<parsed_markdown>")
print(f"\nprompt text before the drawing: {len(PREFIX_TEXT)} characters; rules: {', '.join(lib.CITATIONS)}")

You are a senior piping and facilities engineer checking one engineering drawing of an LNG terminal against the five rules below. The drawing was read from a PDF by a parser: the text of a schematic comes as one block in no particular order, so match items (equipment, tags, labels) by their tags and numbers, not by position; a tag that is not next to its equipment still counts, and the parser may write the letter O as a zero (LOT0-2104A is the tag LOTO-2104A). Report only violations that the text of the drawing itself shows, and do not report a rule that the drawing does not have an element for (for example a sheet with no exits has nothing to check for the egress rule). Check every rule that applies, and every item the rule covers. A drawing can have several violations, including more than one of the same rule, and a compliant drawing has none.

Be exact rather than eager. Before you report a missing item (a LOTO tag, a PSV, a relief schedule entry), search the whole drawing text for 

## Which `responseFormat` works?

The plan writes `responseFormat` as a DDL string, `STRUCT<findings:ARRAY<STRUCT<...>>>`, and wants `failOnError => false` so one bad answer cannot fail the batch. Each form is tried on one drawing (`BRL-INS-PID-0052`, a P&ID with three planted flaws). A bad argument is refused when the function is compiled, before any model call, so a rejected form costs nothing; an accepted one is one Sonnet call. The result of an `ai_query` with `failOnError => false` is a `STRUCT<result: STRING, errorMessage: STRING>`. Each accepted answer is run through `compliance_lib.flatten_judgement`, the same function the pipeline uses.

In [3]:
FINDING_PROPS = {
    "expected_item": {"type": "string"},   # what would make the item comply (LOTO-2104A, PSV-2102, ...), written before the verdict
    "found_in_text": {"type": "boolean"},  # the judge's own search of the drawing text for it
    "status": {"type": "string", "enum": ["FAIL"]},
    "citation": {"type": "string", "enum": lib.CITATIONS},
    "severity": {"type": "string", "enum": list(lib.SEVERITIES)},
    "reason": {"type": "string"},
    "page_num": {"type": "integer"},
    "evidence_text": {"type": "string"},
}
def json_schema(strict, with_enum):
    props = FINDING_PROPS if with_enum else {k: ({"type": v["type"]}) for k, v in FINDING_PROPS.items()}
    item = {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}
    schema = {"type": "object", "properties": {"findings": {"type": "array", "items": item}}, "required": ["findings"], "additionalProperties": False}
    wrapper = {"name": "compliance_findings", "schema": schema}
    if strict:
        wrapper["strict"] = True
    return json.dumps({"type": "json_schema", "json_schema": wrapper})

DDL_FORMAT = "STRUCT<findings:ARRAY<STRUCT<status:STRING,reason:STRING,citation:STRING,severity:STRING,page_num:INT,evidence_text:STRING>>>"
MODEL_PARAMS = "modelParameters => named_struct('temperature', 0.0, 'max_tokens', 4000)"
PROMPT_EXPR = lib.sql_prompt_expr(sql_literal(PREFIX_TEXT))

SIGNATURES = (
    ("DDL string, the plan's form", f"responseFormat => {sql_literal(DDL_FORMAT)}, failOnError => false"),
    ("JSON schema, no strict, no enum", f"responseFormat => {sql_literal(json_schema(False, False))}, failOnError => false"),
    ("JSON schema, strict, with enums", f"responseFormat => {sql_literal(json_schema(True, True))}, failOnError => false"),
    ("JSON schema, strict, with enums + temperature 0", f"responseFormat => {sql_literal(json_schema(True, True))}, failOnError => false, {MODEL_PARAMS}"),
    ("no responseFormat (JSON asked for in the prompt), temperature 0", f"failOnError => false, {MODEL_PARAMS}"),
)
SQL_SAMPLE = f"""
    SELECT CAST(ai_query('{MODEL}', {PROMPT_EXPR}, {{args}}).result AS STRING) AS result
    FROM {PARSED} p JOIN {EXTRACTED} e USING (drawing_id) WHERE p.drawing_id = '{SAMPLE_ID}'
"""
accepted = {}
for what, args in SIGNATURES:
    try:
        answer = spark.sql(SQL_SAMPLE.format(args=args)).first().result
        rows = lib.flatten_judgement(SAMPLE_ID, answer, None)
        accepted[what] = answer
        print(f"accepted  {what}")
        print(f"          -> {len(rows)} row(s): " + "; ".join(f"{r['status']} {r['citation']} ({r['evidence_text']})" if r["status"] != "REVIEW" else f"REVIEW {r['reason'][:150]}" for r in rows))
    except Exception as e:
        print(f"rejected  {what}")
        print(f"          {str(e).strip().splitlines()[0][:260]}")

rejected  DDL string, the plan's form
          [AI_FUNCTION_UNSUPPORTED_RESPONSE_FORMAT.DDL_STRING] Found unsupported response format. AI function: "`ai_query`" does not support the following response format: "STRUCT<findings:ARRAY<STRUCT<status:STRING,reason:STRING,citation:STRING,severity:STRING,page_num
accepted  JSON schema, no strict, no enum
          -> 4 row(s): FAIL OSHA 1910.147 (K-2502 BOG COMPRESSOR); FAIL OSHA 1910.147 (K-2501 REFRIGERANT COMPRESSOR); FAIL ASME B31.3 §322 (V-2504 SURGE DRUM DP 500 PSIG); FAIL ASME B31.3 §322 (V-2506 KNOCKOUT DRUM DP 500 PSIG)
accepted  JSON schema, strict, with enums
          -> 1 row(s): REVIEW Judge output could not be parsed: no findings array
accepted  JSON schema, strict, with enums + temperature 0
          -> 3 row(s): FAIL ASME B16.5 §4.2 (FLG CL150); FAIL OSHA 1910.147 (K-2502); FAIL ASME B31.3 §322 (V-2506)
accepted  no responseFormat (JSON asked for in the prompt), temperature 0
          -> 2 row(s): FAIL OSHA 1910.147 (K-250

Which of these is used is decided by the output above: the pipeline takes the strictest form that was accepted and answered with a parseable result. `failOnError => false` makes a failed call come back as an `errorMessage` instead of failing the query, and `flatten_judgement` turns that, or an answer that is not the expected JSON (a model can wrap JSON in code fences or return prose), into one `REVIEW` row. Temperature 0 keeps a re-run close to this one.

## Load the ground truth into a table

`01_synthetic_data/outputs/ground_truth_findings.jsonl` (written by the drawing generator) holds the 274 flaws it planted: one line per flaw with the page, the box and the citation. It is loaded here into `silver_engineering.ground_truth_findings` for two uses: the left join that fills `matched_ground_truth` and the tight highlight boxes of the demo, and the formal scoring in the next task. **It is never part of any prompt.** `(drawing_id, citation)` is unique, so the join cannot duplicate a finding.

In [4]:
GT_SCHEMA = ("drawing_id string, finding_id string, page_num int, bbox array<double>, citation string, severity string, flaw_type string")
gt_lines = [json.loads(line) for line in GROUND_TRUTH.read_text().splitlines() if line.strip()]
gt_df = spark.createDataFrame(
    [(g["drawing_id"], g["finding_id"], g["page_num"], [float(v) for v in g["bbox"]], g["citation"], g["severity"], g["flaw_type"]) for g in gt_lines],
    GT_SCHEMA,
)
gt_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GROUND_TRUTH_TABLE)
spark.sql(f"COMMENT ON TABLE {GROUND_TRUTH_TABLE} IS 'The flaws the synthetic drawing generator planted (01_synthetic_data/outputs/ground_truth_findings.jsonl), one row per flaw. bbox is [x, y, w, h] as fractions of the page, origin top-left. Used to score the compliance judge; never part of its prompt.'")
spark.sql(f"ALTER TABLE {GROUND_TRUTH_TABLE} SET TBLPROPERTIES ('quality' = 'silver')")

g = spark.sql(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT drawing_id) AS flawed_drawings, COUNT(DISTINCT drawing_id, citation) AS distinct_drawing_citation,
           COUNT_IF(size(bbox) = 4 AND bbox[0] BETWEEN 0 AND 1 AND bbox[1] BETWEEN 0 AND 1) AS bbox_ok
    FROM {GROUND_TRUTH_TABLE}""").first()
print(f"{GROUND_TRUTH_TABLE}: {g.rows} rows on {g.flawed_drawings} drawings, {g.distinct_drawing_citation} distinct (drawing_id, citation), {g.bbox_ok} with a valid box")
assert g.rows == 274 and g.distinct_drawing_citation == 274 and g.bbox_ok == 274
display(spark.table(GROUND_TRUTH_TABLE).groupBy("citation", "severity").count().orderBy("citation"))

mfg_mc_se_sa.bluerridge_silver_engineering.ground_truth_findings: 274 rows on 240 drawings, 274 distinct (drawing_id, citation), 274 with a valid box


citation,severity,count
ASME B16.5 §4.2,critical,63
ASME B31.3 §322,critical,58
ASME B31.3 §341,medium,49
IBC 1005.1,high,45
OSHA 1910.147,high,59


## Review dates

The data is anchored to an as-of date of **2026-11-06**, not to today's date, so each drawing gets a simulated review (submittal) date. It is assigned from the manifest, which says which drawings exist and which carry a flaw, never from what the judge finds: a seeded hash of `drawing_id` picks exactly **238 drawings** for the as-of week `2026-10-31` to `2026-11-06`, **24 of them flawed** (the demo's "238 reviewed, 214 passed, 24 failed this week"), and spreads the other 962 over `2026-06-01` to `2026-10-30`. The judge's recall is not perfect, so the number of drawings it fails in that week will differ somewhat from 24.

In [5]:
manifest = json.loads(MANIFEST.read_text())
review_dates = lib.assign_review_dates(manifest)
in_week = [m for m in manifest if lib.AS_OF_WEEK[0].isoformat() <= review_dates[m["drawing_id"]] <= lib.AS_OF_WEEK[1].isoformat()]
print(f"manifest: {len(manifest)} drawings, {sum(m['has_flaw'] for m in manifest)} flawed")
print(f"in the as-of week {lib.AS_OF_WEEK[0]}..{lib.AS_OF_WEEK[1]}: {len(in_week)} drawings, {sum(m['has_flaw'] for m in in_week)} flawed in the manifest")
print(f"review dates run from {min(review_dates.values())} to {max(review_dates.values())}")
assert len(manifest) == 1200 and len(in_week) == 238 and sum(m["has_flaw"] for m in in_week) == 24
dates_df = spark.createDataFrame(list(review_dates.items()), "drawing_id string, review_date_text string").select(
    "drawing_id", F.to_date("review_date_text").alias("review_date"))
display(dates_df.withColumn("week_start", F.date_trunc("week", "review_date").cast("date")).groupBy("week_start").count().orderBy("week_start"))

manifest: 1200 drawings, 240 flawed
in the as-of week 2026-10-31..2026-11-06: 238 drawings, 24 flawed in the manifest
review dates run from 2026-06-01 to 2026-11-06


week_start,count
2026-06-01,64
2026-06-08,42
2026-06-15,52
2026-06-22,39
2026-06-29,40
2026-07-06,39
2026-07-13,47
2026-07-20,41
2026-07-27,38
2026-08-03,45


## Pick the documents

Every drawing that parsed (`parse_status = 'ok'`), joined to its `drawings_extracted` row on `drawing_id` (never on the title block number, which differs on seven drawings). The judge reads the drawing's type and discipline from there.

In [6]:
docs_all = (
    spark.table(PARSED).where("parse_status = 'ok'")
    .join(spark.table(EXTRACTED).select("drawing_id", "discipline", "revision", "drawing_type"), "drawing_id")
    .select("drawing_id", "parsed_markdown", "discipline", "drawing_type")
)
n_docs_all = docs_all.count()
print(f"drawings parsed ok and extracted: {n_docs_all}")
docs = docs_all
n_docs = n_docs_all
if MAX_ROWS:
    # A fixed subset: the sample drawing, then drawings alternating between flawed and clean ones (per the manifest, used here only
    # to choose which drawings a trial run covers, never in a prompt).
    flaw_flag = spark.createDataFrame([(m["drawing_id"], bool(m["has_flaw"])) for m in manifest], "drawing_id string, has_flaw boolean")
    order = Window.partitionBy("has_flaw").orderBy(F.xxhash64("drawing_id"))
    picked_ids = (docs_all.join(flaw_flag, "drawing_id").withColumn("rn", F.row_number().over(order))
                  .orderBy(F.col("drawing_id") != SAMPLE_ID, "rn", F.col("has_flaw").desc()).limit(MAX_ROWS).select("drawing_id"))
    docs = docs_all.join(picked_ids, "drawing_id")
    n_docs = min(MAX_ROWS, n_docs_all)

# Cost guard 1: the number of model calls is the number of drawings, and the full run is the 1,200-drawing corpus.
expected_calls = n_docs
print(f"drawings to send to the model: {n_docs}; expected judge calls: {expected_calls} (one per drawing)")
assert expected_calls == n_docs
if not MAX_ROWS:
    assert n_docs == 1200, f"the corpus has 1,200 drawings, found {n_docs}: refusing to bill an unexpected number of calls"

drawings parsed ok and extracted: 1200
drawings to send to the model: 1200; expected judge calls: 1200 (one per drawing)


## The `ai_query` call in one plan

Nothing runs yet: a DataFrame is a plan. `repartition` spreads the drawings over several tasks. The answer's two fields are kept as plain strings (`result`, `error`). Before anything is written the physical plan is checked for exactly **one** `AI Query` step, which is one model call per drawing, and the call count was asserted above. (The chosen form is the strict JSON schema with enums plus temperature 0 if the signature test above accepted it.)

In [7]:
CHOSEN = "JSON schema, strict, with enums + temperature 0"
assert CHOSEN in accepted, f"the chosen call form was not accepted above: {list(accepted)}"
CHOSEN_ARGS = dict(SIGNATURES)[CHOSEN]
JUDGE_CALL = f"ai_query('{MODEL}', {PROMPT_EXPR}, {CHOSEN_ARGS})"

ai_df = (
    docs.repartition(PARTITIONS)
    .withColumn("judge", F.expr(JUDGE_CALL))
    .select("drawing_id", F.col("judge.result").alias("result"), F.col("judge.errorMessage").alias("error"), F.current_timestamp().alias("judged_at"))
)
print("columns:", ai_df.columns)

# explain() only plans the query; it does not run it. Only the physical plan is read ("== Photon Explanation ==" quotes the step again).
plan = io.StringIO()
with contextlib.redirect_stdout(plan):
    ai_df.explain(mode="formatted")
physical_plan = re.split(r"^== (?!Physical Plan)", plan.getvalue(), flags=re.M)[0]
ai_steps = re.findall(r"^\(\d+\) AI Query.*$", physical_plan, flags=re.M)
print(f"AI Query steps in the physical plan: {len(ai_steps)}")
assert len(ai_steps) == 1, "the judge should run exactly once per drawing"

columns: ['drawing_id', 'result', 'error', 'judged_at']
AI Query steps in the physical plan: 1


## Send the drawings to the model and write its answers

The only cell that calls the model. `overwrite` makes a re-run replace the table instead of appending a second copy of every answer.

**Run record (cost guard).** The run that produced the outputs below (job run 726850339672013) was a one-time `jobs submit` (`run_type` SUBMIT_RUN) with `"disable_auto_optimization": true` on its task, and made a single attempt (`attempt_number` 0, no retry): 1,200 judge calls, 418 s for the AI step, 576 s for the whole notebook. It is the second full run: the first (job run 880768793395140, 1,200 calls, plainer prompt, see the note under "The rules and the prompt") was kept only as `compliance_judge_raw_v1_baseline` for comparison, and a 10-drawing trial (`max_rows` 10, `*_dryrun` tables, since dropped) came before it. The same flag is set on every task of `resources/job-doc-intelligence.yml`, which has no `max_rows` parameter.

In [8]:
t0 = time.time()
ai_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RAW)
print(f"sent {n_docs} drawings to {MODEL} and wrote {RAW} in {time.time() - t0:.0f} s")
_ = spark.sql(f"COMMENT ON TABLE {RAW} IS 'The compliance judge answers exactly as ai_query returned them (result = the JSON text, error = the error message if the call failed), one row per drawing. compliance_findings is built from this table without further AI calls.'")

sent 1200 drawings to databricks-claude-sonnet-4-5 and wrote mfg_mc_se_sa.bluerridge_gold_pm.compliance_judge_raw in 418 s


## Read the answers back

From here on the notebook reads this table, never the DataFrame above, so nothing is sent to the model again. The shape of the answers: how many calls failed or returned something that is not the expected JSON, and three answers as they came back.

In [9]:
raw = spark.table(RAW)
print(f"{raw.count()} rows in {RAW}")
display(raw.agg(F.count("*").alias("answers"), F.count_if(F.col("error").isNotNull()).alias("call_errors"),
                F.count_if(F.col("result").isNull()).alias("no_result"), F.round(F.avg(F.length("result")), 0).alias("avg_chars")))
display(raw.orderBy(F.rand(seed=42)).limit(3))

1200 rows in mfg_mc_se_sa.bluerridge_gold_pm.compliance_judge_raw


answers,call_errors,no_result,avg_chars
1200,0,0,297.0


drawing_id,result,error,judged_at
BRL-INS-PID-0707,"{""findings"":[{""page_num"":0,""reason"":""Vessel V-2401 (SEPARATOR) has PSV-2401 tag but the set pressure is 785 PSIG, which is not above the vessel design pressure of 785 PSIG. A relief device must be set above the vessel design pressure to provide protection."",""citation"":""ASME B31.3 §322"",""evidence_text"":""PSV-2401 SET 785 PSIG"",""status"":""FAIL"",""found_in_text"":true,""severity"":""critical"",""expected_item"":""PSV-2401""},{""page_num"":0,""reason"":""Vessel V-2403 (KNOCKOUT DRUM) has PSV-2403 tag but the set pressure is 785 PSIG, which is not above the vessel design pressure of 785 PSIG. A relief device must be set above the vessel design pressure to provide protection."",""citation"":""ASME B31.3 §322"",""evidence_text"":""PSV-2403 SET 785 PSIG"",""status"":""FAIL"",""found_in_text"":true,""severity"":""critical"",""expected_item"":""PSV-2403""},{""page_num"":0,""reason"":""Vessel V-2405 (SEPARATOR) has PSV-2405 tag but the set pressure is 1175 PSIG, which is not above the vessel design pressure of 1175 PSIG. A relief device must be set above the vessel design pressure to provide protection."",""citation"":""ASME B31.3 §322"",""evidence_text"":""PSV-2405 SET 1175 PSIG"",""status"":""FAIL"",""found_in_text"":true,""severity"":""critical"",""expected_item"":""PSV-2405""}]}",,2026-11-07T20:36:47.418Z
BRL-PIP-ISO-1198,"{""findings"":[{""page_num"":0,""reason"":""Line 8\""-N2-1176-B3S has design pressure 610 PSIG but uses CL300 flanges rated only 720 PSIG maximum. One flange is shown as CL150 (rated 275 PSIG), which is below the 610 PSIG design pressure."",""citation"":""ASME B16.5 §4.2"",""evidence_text"":""FLG 8\"" CL150 RF WN"",""status"":""FAIL"",""found_in_text"":false,""severity"":""critical"",""expected_item"":""CLASS 600 or higher""}]}",,2026-11-07T20:36:47.418Z
BRL-PIP-PID-0977,"{""findings"":[]}",,2026-11-07T20:36:47.418Z


## Flatten the answers into findings

`compliance_lib.flatten_judgement` turns one answer into rows, and never raises, because one odd answer must not fail a run that has already paid for all the others:

- an `errorMessage`, an empty answer, or text that is not the expected JSON becomes **one `REVIEW` row** with the reason `Judge output could not be parsed: ...`;
- an empty `findings` array becomes **one `PASS` row**; so does an array whose every entry the judge itself marked `found_in_text: true` (see below);
- each finding becomes a **`FAIL` row** unless the judge's own search found the item that makes the drawing comply (`found_in_text` true: the judge wrote down what it expected to find, searched the text, and found it, so the entry is not a violation): the citation is matched to one of the five rules (a finding that cites anything else is not kept; if that leaves nothing, the drawing gets a `REVIEW` row rather than a silent PASS), the severity is coerced to `critical`, `high`, `medium` or `low` (anything else becomes `medium`), and `finding_id` is `<drawing_id>-C01`, `-C02`, ...

It is tried on hand-made answers first, including the bad ones (the same cases as the local unit tests).

In [10]:
good = json.dumps({"findings": [
    {"status": "FAIL", "citation": "ASME B16.5 §4.2", "severity": "critical", "reason": "CL150 on a 500 PSIG line.", "page_num": 0, "evidence_text": "FLG CL150"},
    {"status": "FAIL", "citation": "OSHA 1910.147", "severity": "SEVERE", "reason": "K-2502 has no LOTO tag.", "page_num": 0, "evidence_text": "K-2502"}]})
for name, response, error in (("two findings", good, None), ("no findings", '{"findings": []}', None), ("code fences", "```json\n" + good + "\n```", None),
                              ("not JSON", "I could not read this drawing.", None), ("empty", "", None), ("call failed", None, "REQUEST_LIMIT_EXCEEDED")):
    rows = lib.flatten_judgement("D-TEST", response, error)
    print(f"{name:13} -> {[(r['status'], r['finding_id'], r['citation'], r['severity']) for r in rows]}")
r2 = lib.flatten_judgement("D-TEST", good, None)
assert [r["status"] for r in r2] == ["FAIL", "FAIL"] and r2[1]["severity"] == "medium"
assert [r["status"] for r in lib.flatten_judgement("D-TEST", '{"findings": []}', None)] == ["PASS"]
assert lib.flatten_judgement("D-TEST", "not json", None)[0]["status"] == "REVIEW"
assert lib.flatten_judgement("D-TEST", None, "boom")[0]["reason"].startswith("Judge output could not be parsed")

two findings  -> [('FAIL', 'D-TEST-C01', 'ASME B16.5 §4.2', 'critical'), ('FAIL', 'D-TEST-C02', 'OSHA 1910.147', 'medium')]
no findings   -> [('PASS', 'D-TEST-PASS', None, None)]
code fences   -> [('FAIL', 'D-TEST-C01', 'ASME B16.5 §4.2', 'critical'), ('FAIL', 'D-TEST-C02', 'OSHA 1910.147', 'medium')]
not JSON      -> [('REVIEW', 'D-TEST-REVIEW', None, None)]
empty         -> [('REVIEW', 'D-TEST-REVIEW', None, None)]
call failed   -> [('REVIEW', 'D-TEST-REVIEW', None, None)]


In [11]:
FLAT_SCHEMA = ("drawing_id string, finding_id string, status string, reason string, citation string, severity string, "
               "page_num int, evidence_text string, judged_at timestamp")
# 1,200 short answers: flattened on the driver with the library function (no UDF, so the module need not exist on the workers).
flat_rows = [(*(r[k] for k in ("drawing_id", "finding_id", "status", "reason", "citation", "severity", "page_num", "evidence_text")), a.judged_at)
             for a in spark.table(RAW).collect() for r in lib.flatten_judgement(a.drawing_id, a.result, a.error)]
findings = spark.createDataFrame(flat_rows, FLAT_SCHEMA)
print("one row per finding (the nested array is gone):", findings.columns)
display(findings.groupBy("status").agg(F.count("*").alias("rows"), F.countDistinct("drawing_id").alias("drawings")).orderBy("status"))

one row per finding (the nested array is gone): ['drawing_id', 'finding_id', 'status', 'reason', 'citation', 'severity', 'page_num', 'evidence_text', 'judged_at']


status,rows,drawings
FAIL,627,377
PASS,822,822
REVIEW,1,1


## Where the highlight box comes from (`bbox_source`)

The app draws a rectangle over the drawing for each finding, so a FAIL row needs a page and a box. There are three sources, tried in this order:

1. **`ground_truth`**: the finding matches a planted flaw on `(drawing_id, citation)`. The box and page are the generator's own, which enclose exactly the offending element (a flange label, a table row, an empty slot where a tag belongs). **The demo's tight highlights come from here.** If a drawing has two findings with the same citation, only the first takes the planted box.
2. **`parse_element`**: otherwise, the smallest element in `parse_json` on the judged page whose text contains the judge's `evidence_text`, with its pixel box divided by the page's pixel size and written as `[x, y, w, h]` from the top-left. The sheets are 1224 x 792 pt (17 x 11 in landscape) and `ai_parse_document` renders them at 2.75 px per point, 3366 x 2178 px (checked below against the largest parser box on every drawing).
3. **`page`**: failing that, the whole page, `[0, 0, 1, 1]`.

**What this means in production.** A real drawing has no ground truth, so every highlight would come from source 2 or 3. Those boxes are coarser: a schematic parses as one large `figure` element, holding all its labels as one block, so the best a parse element can do for a flaw inside a P&ID is the figure's box. Table rows (specification sheets) and notes parse as smaller elements and are located much better. The comparison further down measures exactly this on the findings that matched the ground truth.

In [12]:
# Is the page really 3366 x 2178 px? The largest right and bottom edges among all parser boxes must not exceed it, and should come near it
# (the title block sits at the bottom right).
extent = spark.sql(f"""
    SELECT MAX(CAST(el.bbox[0].coord[2] AS DOUBLE)) AS max_x2, MAX(CAST(el.bbox[0].coord[3] AS DOUBLE)) AS max_y2, COUNT(*) AS elements
    FROM {PARSED}
    LATERAL VIEW explode(from_json(get_json_object(parse_json, '$.document.elements'),
        'array<struct<bbox:array<struct<coord:array<double>,page_id:int>>>>')) t AS el
""").first()
W, H = lib.SHEET_PX
print(f"{extent.elements} parser elements; largest x2 = {extent.max_x2:.0f} px (page width {W:.0f}), largest y2 = {extent.max_y2:.0f} px (page height {H:.0f})")
assert extent.max_x2 <= W and extent.max_y2 <= H and extent.max_x2 > 0.9 * W and extent.max_y2 > 0.9 * H

24924 parser elements; largest x2 = 3342 px (page width 3366), largest y2 = 2141 px (page height 2178)


## Resolve the boxes and build `compliance_findings`

Join the findings to the drawing's metadata, to the ground truth (left join on `(drawing_id, citation)`), to the review dates, and to the parsed JSON for the fallback boxes. Only FAIL rows get a page, a box and a `matched_ground_truth` value; PASS and REVIEW rows have NULL in those columns.

In [13]:
gt = (spark.table(GROUND_TRUTH_TABLE).select("drawing_id", "citation", F.col("page_num").alias("gt_page"), F.col("bbox").alias("gt_bbox")))
meta = (spark.table(PARSED).select("drawing_id", "volume_path", "parse_json")
        .join(spark.table(EXTRACTED).select("drawing_id", "discipline", "revision", "drawing_type"), "drawing_id"))

# Findings of one drawing with the same citation: only the first one takes the planted box.
first_of_citation = F.row_number().over(Window.partitionBy("drawing_id", "citation").orderBy("finding_id")) == 1
joined = (findings.join(meta, "drawing_id").join(gt, ["drawing_id", "citation"], "left")
          .withColumn("is_fail", F.col("status") == "FAIL")
          .withColumn("matched", F.when(F.col("is_fail"), F.col("gt_bbox").isNotNull()))
          .withColumn("use_gt", F.col("is_fail") & F.col("gt_bbox").isNotNull() & F.when(F.col("is_fail"), first_of_citation).otherwise(False))
          )

# The parse-element fallback runs on the driver for the FAIL rows that took no planted box (a few hundred small lookups).
need_box = joined.where("is_fail AND NOT use_gt").select("drawing_id", "finding_id", "parse_json", "page_num", "evidence_text").collect()
element_boxes = spark.createDataFrame(
    [(n.drawing_id, n.finding_id, lib.resolve_parse_bbox(n.parse_json, n.page_num, n.evidence_text)) for n in need_box],
    "drawing_id string, finding_id string, element_box array<double>")
joined = joined.join(element_boxes, ["drawing_id", "finding_id"], "left")
print(f"{len(need_box)} FAIL rows without a planted box; a parse element was found for {element_boxes.where('element_box IS NOT NULL').count()}")

final = (joined
    .withColumn("bbox_source", F.when(F.col("use_gt"), "ground_truth").when(F.col("element_box").isNotNull(), "parse_element").when(F.col("is_fail"), "page"))
    .withColumn("bbox_out", F.when(F.col("use_gt"), F.col("gt_bbox")).when(F.col("element_box").isNotNull(), F.col("element_box"))
                .when(F.col("is_fail"), F.array(F.lit(0.0), F.lit(0.0), F.lit(1.0), F.lit(1.0))))
    .withColumn("page_out", F.when(F.col("use_gt"), F.col("gt_page")).when(F.col("is_fail"), F.col("page_num")))
    .join(dates_df, "drawing_id")
    .select(
        "drawing_id", "finding_id", "volume_path",
        F.col("page_out").cast("int").alias("page_num"),
        F.col("bbox_out").cast("array<double>").alias("bbox"),
        "bbox_source", "discipline", "revision", "drawing_type", "status", "severity", "reason", "citation", "evidence_text",
        F.col("matched").alias("matched_ground_truth"),
        "review_date",
        F.col("judged_at").alias("generated_at"),
    ))
print("columns:", final.columns)

380 FAIL rows without a planted box; a parse element was found for 335
columns: ['drawing_id', 'finding_id', 'volume_path', 'page_num', 'bbox', 'bbox_source', 'discipline', 'revision', 'drawing_type', 'status', 'severity', 'reason', 'citation', 'evidence_text', 'matched_ground_truth', 'review_date', 'generated_at']


In [14]:
t0 = time.time()
final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"wrote {TARGET} in {time.time() - t0:.0f} s")

spark.sql(
    f"COMMENT ON TABLE {TARGET} IS 'One row per compliance finding from the ai_query judge (databricks-claude-sonnet-4-5) over the 1,200 drawings. Every drawing has at least one row: "
    "N FAIL rows, or one PASS row, or one REVIEW row when the judge answer could not be used. Built from compliance_judge_raw by 03_compliance_check.'"
)
COLUMN_COMMENTS = {
    "drawing_id": "Drawing id; key into drawings_extracted. With finding_id the primary key",
    "finding_id": "<drawing_id>-C01, -C02, ... for FAIL rows; <drawing_id>-PASS or <drawing_id>-REVIEW for the one-row cases",
    "volume_path": "Unity Catalog Volume path of the PDF",
    "page_num": "0-based page of the violation; NULL on PASS and REVIEW rows",
    "bbox": "[x, y, w, h] as fractions (0-1) of the page, origin top-left; NULL on PASS and REVIEW rows",
    "bbox_source": "ground_truth (the generator's planted box), parse_element (smallest parsed element holding evidence_text) or page ([0,0,1,1]); NULL on PASS and REVIEW rows",
    "discipline": "From drawings_extracted", "revision": "From drawings_extracted", "drawing_type": "From drawings_extracted",
    "status": "PASS, FAIL or REVIEW (REVIEW: the judge answer was an error or could not be parsed)",
    "severity": "critical, high, medium or low (anything else the judge said is medium); NULL unless FAIL",
    "reason": "Short explanation a project manager can read",
    "citation": "The rule violated: ASME B16.5 §4.2, OSHA 1910.147, IBC 1005.1, ASME B31.3 §341 or ASME B31.3 §322; NULL unless FAIL",
    "evidence_text": "The judge's short quote from the drawing text that shows the violation; NULL unless FAIL",
    "matched_ground_truth": "TRUE when the synthetic generator planted a flaw with this citation on this drawing; NULL unless FAIL",
    "review_date": "Simulated review date; the as-of week 2026-10-31..2026-11-06 holds 238 drawings",
    "generated_at": "When the judge step ran",
}
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN {column} COMMENT {sql_literal(comment)}")
spark.sql(f"ALTER TABLE {TARGET} SET TBLPROPERTIES ('quality' = 'gold')")
display(spark.sql(f"""
    SELECT ordinal_position AS position, column_name, full_data_type AS data_type
    FROM {catalog}.information_schema.columns
    WHERE table_schema = '{prefix}_gold_pm' AND table_name = '{TARGET.split('.')[-1]}'
    ORDER BY ordinal_position
"""))

wrote mfg_mc_se_sa.bluerridge_gold_pm.compliance_findings in 4 s


position,column_name,data_type
0,drawing_id,string
1,finding_id,string
2,volume_path,string
3,page_num,int
4,bbox,array<double>
5,bbox_source,string
6,discipline,string
7,revision,string
8,drawing_type,string
9,status,string


## Verify

The table's shape and the row rules as assertions: the ruled schema, a primary key of `(drawing_id, finding_id)`, every drawing sent has a row and none is an orphan against `drawings_extracted`, a drawing is exactly one of {FAIL rows only, one PASS row, one REVIEW row}, FAIL rows have a box inside 0-1, one of the five citations and one of the four severities, PASS and REVIEW rows have NULL in the FAIL-only columns, every REVIEW row has a reason, and (full run) exactly 238 drawings in the as-of week.

In [15]:
EXPECTED_SCHEMA = [
    ("drawing_id", "string"), ("finding_id", "string"), ("volume_path", "string"), ("page_num", "int"), ("bbox", "array<double>"),
    ("bbox_source", "string"), ("discipline", "string"), ("revision", "string"), ("drawing_type", "string"), ("status", "string"),
    ("severity", "string"), ("reason", "string"), ("citation", "string"), ("evidence_text", "string"), ("matched_ground_truth", "boolean"),
    ("review_date", "date"), ("generated_at", "timestamp"),
]
assert spark.table(TARGET).dtypes == EXPECTED_SCHEMA, spark.table(TARGET).dtypes

cites_sql = ", ".join(sql_literal(c) for c in lib.CITATIONS)
checks = spark.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT drawing_id, finding_id) AS distinct_keys,
           COUNT(DISTINCT drawing_id) AS drawings,
           COUNT_IF(status NOT IN ('PASS', 'FAIL', 'REVIEW')) AS bad_status,
           COUNT_IF(status = 'FAIL' AND (bbox IS NULL OR size(bbox) <> 4 OR NOT forall(bbox, v -> v BETWEEN 0 AND 1) OR page_num IS NULL OR bbox_source IS NULL)) AS fail_bad_box,
           COUNT_IF(status = 'FAIL' AND bbox_source NOT IN ('ground_truth', 'parse_element', 'page')) AS bad_source,
           COUNT_IF(status = 'FAIL' AND (citation IS NULL OR citation NOT IN ({cites_sql}))) AS fail_bad_citation,
           COUNT_IF(status = 'FAIL' AND (severity IS NULL OR severity NOT IN ('critical', 'high', 'medium', 'low'))) AS fail_bad_severity,
           COUNT_IF(status = 'FAIL' AND (matched_ground_truth IS NULL OR reason IS NULL OR evidence_text IS NULL)) AS fail_missing_text,
           COUNT_IF(status <> 'FAIL' AND (citation IS NOT NULL OR page_num IS NOT NULL OR bbox IS NOT NULL OR bbox_source IS NOT NULL
                                           OR severity IS NOT NULL OR matched_ground_truth IS NOT NULL)) AS sentinel_not_null,
           COUNT_IF(status = 'REVIEW' AND reason IS NULL) AS review_no_reason,
           COUNT_IF(review_date IS NULL OR generated_at IS NULL OR volume_path IS NULL) AS missing_context
    FROM {TARGET}
""")
display(checks)
c = checks.first()
shape = spark.sql(f"""
    SELECT drawing_id, COUNT(*) AS n, COUNT_IF(status = 'FAIL') AS fails, COUNT_IF(status = 'PASS') AS passes, COUNT_IF(status = 'REVIEW') AS reviews
    FROM {TARGET} GROUP BY drawing_id
""")
bad_shape = shape.where("NOT ((fails >= 1 AND passes = 0 AND reviews = 0) OR (n = 1 AND passes = 1) OR (n = 1 AND reviews = 1))").count()
orphans = spark.sql(f"SELECT COUNT(*) AS n FROM {TARGET} t WHERE NOT EXISTS (SELECT 1 FROM {EXTRACTED} e WHERE e.drawing_id = t.drawing_id)").first().n
unreviewed = docs_all.join(spark.table(TARGET).select("drawing_id").distinct(), "drawing_id", "left_anti").join(docs.select("drawing_id"), "drawing_id").count()
print(f"{c.rows} rows, {c.distinct_keys} distinct (drawing_id, finding_id), {c.drawings} drawings; {bad_shape} drawings with a wrong mix of statuses; "
      f"{orphans} rows without a drawings_extracted row; {unreviewed} drawings sent but without a row")

assert c.rows == c.distinct_keys, "(drawing_id, finding_id) is not unique"
assert c.drawings == n_docs and unreviewed == 0, "every drawing sent must have at least one row"
assert orphans == 0 and bad_shape == 0 and c.bad_status == 0
assert c.fail_bad_box == 0 and c.bad_source == 0 and c.fail_bad_citation == 0 and c.fail_bad_severity == 0 and c.fail_missing_text == 0
assert c.sentinel_not_null == 0 and c.review_no_reason == 0 and c.missing_context == 0
if not MAX_ROWS:
    assert c.drawings == 1200, "the corpus has 1,200 drawings"
    week = spark.sql(f"SELECT COUNT(DISTINCT drawing_id) AS n FROM {TARGET} WHERE review_date BETWEEN DATE'2026-10-31' AND DATE'2026-11-06'").first().n
    assert week == 238, week
    review_share = spark.sql(f"SELECT COUNT(DISTINCT drawing_id) / {n_docs} AS s FROM {TARGET} WHERE status = 'REVIEW'").first().s
    assert review_share <= 0.05, f"more than 5% of the drawings ended in REVIEW ({review_share:.1%}); read compliance_judge_raw.error"
print("all checks passed")

rows,distinct_keys,drawings,bad_status,fail_bad_box,bad_source,fail_bad_citation,fail_bad_severity,fail_missing_text,sentinel_not_null,review_no_reason,missing_context
1450,1450,1200,0,0,0,0,0,0,0,0,0


1450 rows, 1450 distinct (drawing_id, finding_id), 1200 drawings; 0 drawings with a wrong mix of statuses; 0 rows without a drawings_extracted row; 0 drawings sent but without a row
all checks passed


## What a project manager sees

The headline numbers, computed from `compliance_findings` alone: how many drawings were reviewed and how they came out, which rules were violated and how often, where the boxes came from, and the as-of week.

In [16]:
# One status per drawing: FAIL if it has any FAIL row, else REVIEW, else PASS.
drawing_status = spark.sql(f"""
    SELECT drawing_id, MIN(review_date) AS review_date,
           CASE WHEN COUNT_IF(status = 'FAIL') > 0 THEN 'FAIL' WHEN COUNT_IF(status = 'REVIEW') > 0 THEN 'REVIEW' ELSE 'PASS' END AS drawing_status,
           COUNT_IF(status = 'FAIL') AS findings
    FROM {TARGET} GROUP BY drawing_id
""")
drawing_status.createOrReplaceTempView("drawing_status")
print("drawings reviewed, and how many came out PASS / FAIL / REVIEW:")
display(spark.sql("""
    SELECT COUNT(*) AS drawings_reviewed, COUNT_IF(drawing_status = 'PASS') AS passed, COUNT_IF(drawing_status = 'FAIL') AS failed,
           COUNT_IF(drawing_status = 'REVIEW') AS needs_review, SUM(findings) AS fail_rows
    FROM drawing_status"""))
print("FAIL rows by citation (and how many matched a planted flaw):")
display(spark.sql(f"""
    SELECT citation, COUNT(*) AS fail_rows, COUNT(DISTINCT drawing_id) AS drawings, COUNT_IF(matched_ground_truth) AS matched_ground_truth,
           COUNT_IF(severity = 'critical') AS critical, COUNT_IF(severity = 'high') AS high, COUNT_IF(severity = 'medium') AS medium, COUNT_IF(severity = 'low') AS low
    FROM {TARGET} WHERE status = 'FAIL' GROUP BY citation ORDER BY fail_rows DESC"""))
print("bbox_source of the FAIL rows:")
display(spark.sql(f"""
    SELECT bbox_source, COUNT(*) AS fail_rows, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct FROM {TARGET} WHERE status = 'FAIL' GROUP BY bbox_source ORDER BY fail_rows DESC"""))
m = spark.sql(f"SELECT COUNT(*) AS fail_rows, COUNT_IF(matched_ground_truth) AS matched FROM {TARGET} WHERE status = 'FAIL'").first()
print(f"matched_ground_truth on FAIL rows: {m.matched} of {m.fail_rows} ({m.matched / max(m.fail_rows, 1):.1%})")
print("as-of week 2026-10-31..2026-11-06 (review_date), the demo's \"reviewed / passed / failed this week\":")
display(spark.sql("""
    SELECT COUNT(*) AS reviewed, COUNT_IF(drawing_status = 'PASS') AS passed, COUNT_IF(drawing_status = 'FAIL') AS failed, COUNT_IF(drawing_status = 'REVIEW') AS needs_review
    FROM drawing_status WHERE review_date BETWEEN DATE'2026-10-31' AND DATE'2026-11-06'"""))
print("status of the drawings by drawing_type:")
display(spark.sql("SELECT e.drawing_type, COUNT(*) AS drawings, COUNT_IF(d.drawing_status = 'FAIL') AS failed FROM drawing_status d JOIN " + EXTRACTED + " e USING (drawing_id) GROUP BY e.drawing_type ORDER BY drawings DESC"))

drawings reviewed, and how many came out PASS / FAIL / REVIEW:


drawings_reviewed,passed,failed,needs_review,fail_rows
1200,822,377,1,627


FAIL rows by citation (and how many matched a planted flaw):


citation,fail_rows,drawings,matched_ground_truth,critical,high,medium,low
OSHA 1910.147,323,189,69,0,323,0,0
ASME B31.3 §322,110,85,70,110,0,0,0
ASME B16.5 §4.2,99,82,54,99,0,0,0
ASME B31.3 §341,50,49,50,0,0,50,0
IBC 1005.1,45,45,42,0,45,0,0


bbox_source of the FAIL rows:


bbox_source,fail_rows,pct
parse_element,335,53.4
ground_truth,247,39.4
page,45,7.2


matched_ground_truth on FAIL rows: 285 of 627 (45.5%)
as-of week 2026-10-31..2026-11-06 (review_date), the demo's "reviewed / passed / failed this week":


reviewed,passed,failed,needs_review
238,186,52,0


status of the drawings by drawing_type:


drawing_type,drawings,failed
ISO,397,91
PID,373,169
GA,269,64
SPEC,135,53
OTHER,26,0


## Against the planted flaws (approximate)

The formal scoring is the next task; this is a first look. A finding counts as correct when the drawing has a planted flaw with the same citation (`matched_ground_truth`). **Recall** is the share of the 274 planted flaws that the judge found (on `(drawing_id, citation)`), **precision** the share of its distinct `(drawing_id, citation)` findings that are planted flaws. On a trial run only the drawings sent are counted.

In [17]:
scored = spark.sql(f"""
    WITH judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL'),
         sent AS (SELECT DISTINCT drawing_id FROM {TARGET}),
         planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g JOIN sent USING (drawing_id))
    SELECT (SELECT COUNT(*) FROM planted) AS planted_flaws,
           (SELECT COUNT(*) FROM judged) AS judged_findings,
           (SELECT COUNT(*) FROM judged j JOIN planted p USING (drawing_id, citation)) AS true_positives
""").first()
print(f"planted flaws on the drawings sent: {scored.planted_flaws}; distinct judged findings: {scored.judged_findings}; found correctly: {scored.true_positives}")
print(f"recall {scored.true_positives / max(scored.planted_flaws, 1):.1%}, precision {scored.true_positives / max(scored.judged_findings, 1):.1%}")
display(spark.sql(f"""
    WITH planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g WHERE g.drawing_id IN (SELECT drawing_id FROM {TARGET})),
         judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL')
    SELECT COALESCE(p.citation, j.citation) AS citation, COUNT(p.citation) AS planted, COUNT(j.citation) AS judged,
           COUNT_IF(p.citation IS NOT NULL AND j.citation IS NOT NULL) AS found, COUNT_IF(p.citation IS NULL) AS false_alarms, COUNT_IF(j.citation IS NULL) AS missed
    FROM planted p FULL OUTER JOIN judged j ON p.drawing_id = j.drawing_id AND p.citation = j.citation
    GROUP BY COALESCE(p.citation, j.citation) ORDER BY citation"""))
print("false alarms and misses by drawing type:")
display(spark.sql(f"""
    WITH planted AS (SELECT g.drawing_id, g.citation FROM {GROUND_TRUTH_TABLE} g WHERE g.drawing_id IN (SELECT drawing_id FROM {TARGET})),
         judged AS (SELECT DISTINCT drawing_id, citation FROM {TARGET} WHERE status = 'FAIL')
    SELECT e.drawing_type, COUNT_IF(p.citation IS NOT NULL AND j.citation IS NOT NULL) AS found, COUNT_IF(p.citation IS NULL) AS false_alarms, COUNT_IF(j.citation IS NULL) AS missed
    FROM planted p FULL OUTER JOIN judged j ON p.drawing_id = j.drawing_id AND p.citation = j.citation
    JOIN {EXTRACTED} e ON e.drawing_id = COALESCE(p.drawing_id, j.drawing_id)
    GROUP BY e.drawing_type ORDER BY e.drawing_type"""))

planted flaws on the drawings sent: 274; distinct judged findings: 450; found correctly: 247
recall 90.1%, precision 54.9%


citation,planted,judged,found,false_alarms,missed
ASME B16.5 §4.2,63,82,52,30,11
ASME B31.3 §322,58,85,53,32,5
ASME B31.3 §341,49,49,49,0,0
IBC 1005.1,45,45,42,3,3
OSHA 1910.147,59,189,51,138,8


false alarms and misses by drawing type:


drawing_type,found,false_alarms,missed
GA,59,12,8
ISO,59,45,1
PID,92,121,18
SPEC,37,25,0


## How good would the production highlight be?

For the findings that matched a planted flaw, the table holds the planted box. What the `parse_element` rule would have produced for the same finding is computed here, to show how much coarser it is: the share of those findings for which it finds an element at all, the share whose box contains the planted one, and the area of that box compared with the planted box (1.0 = as tight; a `figure` holding a whole schematic is 100 times larger or more).

In [18]:
matched_rows = (spark.table(TARGET).where("status = 'FAIL' AND bbox_source = 'ground_truth'")
                .join(spark.table(PARSED).select("drawing_id", "parse_json"), "drawing_id")
                .select("drawing_id", "drawing_type", "citation", "page_num", "bbox", "evidence_text", "parse_json").collect())
comparison = {}
for r in matched_rows:
    box = lib.resolve_parse_bbox(r.parse_json, r.page_num, r.evidence_text)
    gx, gy, gw, gh = r.bbox
    entry = comparison.setdefault(r.drawing_type, {"findings": 0, "located": 0, "contains": 0, "ratios": []})
    entry["findings"] += 1
    if box:
        entry["located"] += 1
        x, y, w, h = box
        if x - 0.002 <= gx and y - 0.002 <= gy and x + w + 0.002 >= gx + gw and y + h + 0.002 >= gy + gh:
            entry["contains"] += 1
        entry["ratios"].append((w * h) / max(gw * gh, 1e-9))
rows = []
for dtype, e in sorted(comparison.items()):
    ratios = sorted(e["ratios"])
    rows.append((dtype, e["findings"], e["located"], e["contains"], round(ratios[len(ratios) // 2], 1) if ratios else None))
display(spark.createDataFrame(rows, "drawing_type string, matched_findings int, element_found int, element_contains_planted_box int, median_area_vs_planted double"))
tot = [sum(x[i] for x in rows) for i in (1, 2, 3)]
print(f"{tot[0]} matched findings: a parse element holds the evidence text for {tot[1]}, and its box contains the planted box for {tot[2]}")

drawing_type,matched_findings,element_found,element_contains_planted_box,median_area_vs_planted
GA,59,33,33,93.0
ISO,59,59,59,140.6
PID,92,90,40,6.5
SPEC,37,11,5,6.3


247 matched findings: a parse element holds the evidence text for 193, and its box contains the planted box for 137


## A sample for the repository

Twenty rows written to `outputs/sample_findings.jsonl`, one JSON object per line: FAIL rows first (two per citation where the judge found that many, so all five rules appear), then PASS rows, and every REVIEW row there is (up to three), filled up with more FAIL rows.

In [19]:
import datetime
final_rows = spark.table(TARGET).orderBy("drawing_id", "finding_id").collect()
fails = [r for r in final_rows if r.status == "FAIL"]
passes = [r for r in final_rows if r.status == "PASS"]
reviews = [r for r in final_rows if r.status == "REVIEW"][:3]

picked = []
for citation in lib.CITATIONS:  # up to two FAIL rows per rule, one of them matched to the ground truth when possible
    of_rule = [r for r in fails if r.citation == citation]
    first = next((r for r in of_rule if r.matched_ground_truth and r.bbox_source == "ground_truth"), None)
    chosen = ([first] if first else []) + [r for r in of_rule if r is not first][:2 - bool(first)]
    picked += chosen
picked += reviews
picked += passes[:8]
picked_keys = {(r.drawing_id, r.finding_id) for r in picked}
picked += [r for r in fails if (r.drawing_id, r.finding_id) not in picked_keys][: max(20 - len(picked), 0)]
picked = picked[:20]
if not MAX_ROWS:
    assert len(picked) == 20 and {r.citation for r in picked if r.status == "FAIL"} == set(lib.CITATIONS), "expected 20 rows covering all five rules"

utc = spark.conf.get("spark.sql.session.timeZone") in ("UTC", "Etc/UTC")  # serverless sessions run in UTC
lines = []
for r in picked:
    d = r.asDict()
    d["bbox"] = list(r.bbox) if r.bbox is not None else None
    d["review_date"] = r.review_date.isoformat()
    d["generated_at"] = r.generated_at.strftime("%Y-%m-%dT%H:%M:%S") + ("Z" if utc else "")
    lines.append(json.dumps(d, ensure_ascii=False))
OUT.mkdir(exist_ok=True)
(OUT / "sample_findings.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {OUT / 'sample_findings.jsonl'}: {len(lines)} rows ({sum(r.status == 'FAIL' for r in picked)} FAIL, {sum(r.status == 'PASS' for r in picked)} PASS, {sum(r.status == 'REVIEW' for r in picked)} REVIEW)\n")
print("\n".join(lines[:6]))

wrote outputs/sample_findings.jsonl: 20 rows (11 FAIL, 8 PASS, 1 REVIEW)

{"drawing_id": "BRL-INS-PID-0052", "finding_id": "BRL-INS-PID-0052-C01", "volume_path": "/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-INS-PID-0052.pdf", "page_num": 0, "bbox": [0.235, 0.5532, 0.0334, 0.0365], "bbox_source": "ground_truth", "discipline": "instr", "revision": "B", "drawing_type": "PID", "status": "FAIL", "severity": "critical", "reason": "Line 10-BOG-2551-B3S has design pressure 500 PSIG and uses flange class CL150, whose maximum allowable pressure is 275 PSIG, below the design pressure.", "citation": "ASME B16.5 §4.2", "evidence_text": "FLG CL150", "matched_ground_truth": true, "review_date": "2026-07-12", "generated_at": "2026-11-07T20:36:47Z"}
{"drawing_id": "BRL-INS-PID-0055", "finding_id": "BRL-INS-PID-0055-C01", "volume_path": "/Volumes/mfg_mc_se_sa/bluerridge_landing/drawings/BRL-INS-PID-0055.pdf", "page_num": 0, "bbox": [0.2299, 0.2501, 0.0334, 0.0365], "bbox_source": "ground_truth